# ================================================
# SILVER LAYER — Cleaning & Enrichment
# ================================================

In [0]:
from pyspark.sql.functions import (
    col, hour, dayofweek, month, round,
    unix_timestamp, when, current_timestamp
)

In [0]:
# Read from Bronze table
silver_df = spark.read.table("nyc_taxi_catalog.bronze.bronze_taxi_data")

## # Step 1 — Fix data types

In [0]:
silver_df = silver_df \
    .withColumn("tpep_pickup_datetime",
                col("tpep_pickup_datetime").cast("timestamp")) \
    .withColumn("tpep_dropoff_datetime",
                col("tpep_dropoff_datetime").cast("timestamp"))

## # Step 2 — Remove invalid records

In [0]:
silver_df = silver_df.filter(
    (col("fare_amount") > 0) &
    (col("trip_distance") > 0) &
    (col("passenger_count") > 0) &
    (col("passenger_count") <= 6) &
    (col("fare_amount") <= 500) &
    (col("trip_distance") <= 100)
)

## # Step 3 — Remove nulls

In [0]:
silver_df = silver_df.dropna(subset=[
    "passenger_count", "fare_amount",
    "trip_distance", "tpep_pickup_datetime"
])

## # Step 4 — Add new columns

In [0]:
silver_df = silver_df \
    .withColumn("trip_duration_mins",
        round((unix_timestamp("tpep_dropoff_datetime") -
               unix_timestamp("tpep_pickup_datetime")) / 60, 2)) \
    .withColumn("cost_per_mile",
        round(col("fare_amount") / col("trip_distance"), 2)) \
    .withColumn("pickup_hour",
        hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day",
        dayofweek("tpep_pickup_datetime")) \
    .withColumn("trip_category",
        when(col("trip_distance") < 2, "Short")
        .when(col("trip_distance") < 10, "Medium")
        .otherwise("Long")) \
    .withColumn("payment_label",
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No Charge")
        .when(col("payment_type") == 4, "Dispute")
        .otherwise("Unknown")) \
    .withColumn("is_high_tip",
        when(col("tip_amount") > col("fare_amount") * 0.2,
             "Yes").otherwise("No")) \
    .withColumn("silver_timestamp", current_timestamp())


## # Step 5 — Remove impossible durations

In [0]:
silver_df = silver_df.filter(
    (col("trip_duration_mins") >= 1) &
    (col("trip_duration_mins") <= 1440)
)

## # Save as Delta Table in Silver Schema

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_catalog.silver.silver_taxi_data")

print(f"✅ Silver Layer Done! Records: {silver_df.count():,}")

In [0]:
display(silver_df.limit(5))